# 04 · Analysis — answering the four questions from Gold only

Every query below reads **only** `gold.*` tables (the fact, the four dimensions and the genre bridge).
Each question has: the SQL, the result table, a chart, and an evidence-based interpretation.

| # | Question | Dimension(s) | Requirement covered |
|---|---|---|---|
| Q1 | How did the number of ratings and the average rating change year by year (1995–2019)? | `dim_date` | aggregate (count, average) over time |
| Q2 | Which genres receive the highest and lowest average ratings? | `dim_genre` via bridge | ordered ranking |
| Q3 | Are films from older release decades rated higher than recent films? | `dim_movie` | comparison |
| Q4 | How do casual, regular, heavy and power users differ in how much they rate and how generously? | `dim_user` | comparison + share of total |

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
from IPython.display import Markdown, display

from lakehouse import BLUE, FIG_DIR, INK_SECONDARY, ORANGE, apply_chart_style, connect

apply_chart_style()
FIG_DIR.mkdir(exist_ok=True)
con = connect()

MIN_RATINGS = 10_000   # minimum ratings for a genre / decade to be ranked (avoids tiny-sample averages)

overall = con.sql('''
    SELECT count(*) AS n, avg(rating) AS avg_rating, avg(is_positive) AS pct_pos
    FROM gold.fact_rating
''').df().iloc[0]
first_day, last_day = con.sql('''
    SELECT min(d.full_date), max(d.full_date)
    FROM gold.fact_rating f JOIN gold.dim_date d USING (date_key)
''').fetchone()
print(f"{overall.n:,.0f} ratings | overall average {overall.avg_rating:.3f} stars | "
      f"{100 * overall.pct_pos:.1f}% are 4+ stars | {first_day} → {last_day}")

---
## Q1 · How did the number of ratings and the average rating change year by year?

In [ ]:
Q1 = '''
SELECT d.year,
       count(*)                          AS n_ratings,
       count(DISTINCT f.user_key)        AS active_users,
       round(avg(f.rating), 3)           AS avg_rating,
       round(100 * avg(f.is_positive), 1) AS pct_positive
FROM gold.fact_rating f
JOIN gold.dim_date d ON d.date_key = f.date_key
GROUP BY d.year
ORDER BY d.year
'''
q1 = con.sql(Q1).df()
display(q1)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True,
                               gridspec_kw={"height_ratios": [3, 2], "hspace": 0.35})
ax1.bar(q1["year"], q1["n_ratings"] / 1e6, color=BLUE, width=0.7)
ax1.set_title("Ratings submitted per year (millions)")
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:g}M"))
ax1.grid(axis="x", visible=False)

ax2.plot(q1["year"], q1["avg_rating"], color=BLUE, marker="o", markersize=5)
ax2.axhline(overall.avg_rating, color=INK_SECONDARY, linestyle="--", linewidth=1)
ax2.text(q1["year"].min(), overall.avg_rating, f" overall {overall.avg_rating:.2f}",
         va="bottom", color=INK_SECONDARY, fontsize=9)
ax2.set_title("Average rating per year (stars)")
ax2.set_xlabel(f"Year the rating was given (UTC). {first_day.year} and {last_day.year} are partial years "
               f"(data runs {first_day} → {last_day}).")
ax2.xaxis.set_major_locator(mtick.MultipleLocator(2))
fig.savefig(FIG_DIR / "q1_ratings_by_year.png")
plt.show()

In [ ]:
full = q1[(q1["year"] > first_day.year) & (q1["year"] < last_day.year)]   # complete years only
peak = full.loc[full["n_ratings"].idxmax()]
low = full.loc[full["n_ratings"].idxmin()]
hi_avg = full.loc[full["avg_rating"].idxmax()]
lo_avg = full.loc[full["avg_rating"].idxmin()]
start, end = full.iloc[0], full.iloc[-1]
display(Markdown(f'''
**Evidence (complete years {int(start.year)}–{int(end.year)}):**
- Busiest year: **{int(peak.year)}** with **{peak.n_ratings:,.0f}** ratings from {peak.active_users:,.0f} active users;
  quietest: {int(low.year)} with {low.n_ratings:,.0f}.
- Average rating ranged from **{lo_avg.avg_rating:.3f}** ({int(lo_avg.year)}) to **{hi_avg.avg_rating:.3f}** ({int(hi_avg.year)}),
  a spread of {hi_avg.avg_rating - lo_avg.avg_rating:.2f} stars; it moved from {start.avg_rating:.3f} in {int(start.year)}
  to {end.avg_rating:.3f} in {int(end.year)}.
- Share of 4+ star ratings: {start.pct_positive:.1f}% in {int(start.year)} vs {end.pct_positive:.1f}% in {int(end.year)}.
'''))

**Interpretation.** > ✏️ TODO after the full run: 2–4 sentences explaining what the evidence above means for the question.

---
## Q2 · Which genres receive the highest and lowest average ratings? (ranked)

A rating counts once in each genre of its movie. `IMAX` (a format) is excluded, and only genres with at least
`MIN_RATINGS` ratings are ranked.

In [ ]:
Q2 = f'''
SELECT rank() OVER (ORDER BY avg(f.rating) DESC) AS genre_rank,
       g.genre_name,
       count(*)                           AS n_ratings,
       count(DISTINCT b.movie_key)        AS n_movies_rated,
       round(avg(f.rating), 3)            AS avg_rating,
       round(100 * avg(f.is_positive), 1) AS pct_positive
FROM gold.fact_rating f
JOIN gold.bridge_movie_genre b ON b.movie_key = f.movie_key
JOIN gold.dim_genre g          ON g.genre_key = b.genre_key
WHERE g.is_content_genre
GROUP BY g.genre_name
HAVING count(*) >= {MIN_RATINGS}
ORDER BY genre_rank
'''
q2 = con.sql(Q2).df()
display(q2)

In [ ]:
d = q2.sort_values("avg_rating")
fig, ax = plt.subplots(figsize=(9, 0.38 * len(d) + 1.2))
ax.hlines(d["genre_name"], d["avg_rating"].min() - 0.05, d["avg_rating"], color=BLUE, linewidth=2, alpha=0.35)
ax.plot(d["avg_rating"], d["genre_name"], "o", color=BLUE, markersize=8)
ax.axvline(overall.avg_rating, color=INK_SECONDARY, linestyle="--", linewidth=1)
ax.text(overall.avg_rating, len(d) - 0.35, f" overall {overall.avg_rating:.2f}", color=INK_SECONDARY, fontsize=9)
for _, r in d.iterrows():
    ax.text(r.avg_rating + 0.01, r.genre_name, f"{r.avg_rating:.2f}", va="center", fontsize=8, color=INK_SECONDARY)
ax.set_title("Average rating by genre (stars), ranked")
ax.set_xlabel(f"Average rating — genres with ≥ {MIN_RATINGS:,} ratings; axis does not start at 0")
ax.grid(axis="y", visible=False)
fig.savefig(FIG_DIR / "q2_genre_ranking.png")
plt.show()

In [ ]:
top, bottom = q2.iloc[0], q2.iloc[-1]
most = q2.loc[q2["n_ratings"].idxmax()]
display(Markdown(f'''
**Evidence ({len(q2)} genres ranked):**
- Highest: **{top.genre_name}** at **{top.avg_rating:.3f}** stars ({top.n_ratings:,.0f} ratings, {top.pct_positive:.1f}% positive).
- Lowest: **{bottom.genre_name}** at **{bottom.avg_rating:.3f}** ({bottom.n_ratings:,.0f} ratings, {bottom.pct_positive:.1f}% positive).
- Gap between top and bottom genre: {top.avg_rating - bottom.avg_rating:.2f} stars.
- Most-rated genre: {most.genre_name} ({most.n_ratings:,.0f} ratings), ranked #{int(most["genre_rank"])}.
'''))

**Interpretation.** > ✏️ TODO after the full run: 2–4 sentences explaining what the evidence above means for the question.

---
## Q3 · Are films from older release decades rated higher than recent films?

Compares rating-weighted average stars by the decade the film was **released** (from `dim_movie`), against the
overall average. Films with no release year are shown as `Unknown` in the table but left out of the chart.

In [ ]:
Q3 = f'''
SELECT m.release_decade,
       m.release_decade_start,
       count(DISTINCT f.movie_key)         AS n_movies_rated,
       count(*)                            AS n_ratings,
       round(avg(f.rating), 3)             AS avg_rating,
       round(avg(f.rating) - (SELECT avg(rating) FROM gold.fact_rating), 3) AS diff_vs_overall,
       round(100 * avg(f.is_positive), 1)  AS pct_positive
FROM gold.fact_rating f
JOIN gold.dim_movie m ON m.movie_key = f.movie_key
GROUP BY m.release_decade, m.release_decade_start
ORDER BY m.release_decade_start NULLS LAST
'''
q3 = con.sql(Q3).df()
display(q3)

In [ ]:
d = q3[q3["release_decade_start"].notna() & (q3["n_ratings"] >= MIN_RATINGS)]
fig, ax = plt.subplots(figsize=(10, 4.5))
def fmt_count(n):
    return f"{n / 1e6:.1f}M" if n >= 1e6 else f"{n / 1e3:.0f}k"

labels = [f"{dec}\n{fmt_count(n)}" for dec, n in zip(d["release_decade"], d["n_ratings"])]
x = np.arange(len(d))
ax.plot(x, d["avg_rating"], color=BLUE, marker="o", markersize=8)
ax.axhline(overall.avg_rating, color=INK_SECONDARY, linestyle="--", linewidth=1)
ax.text(x[-1] + 0.4, overall.avg_rating, f"overall {overall.avg_rating:.2f}", va="bottom", ha="left",
        color=INK_SECONDARY, fontsize=9)
ax.set_xticks(x, labels)
ax.set_xlim(-0.5, x[-1] + 1.4)
ax.set_title("Average rating by film release decade (stars)")
ax.set_xlabel(f"Release decade (number of ratings below each label); decades with ≥ {MIN_RATINGS:,} ratings")
ax.grid(axis="x", visible=False)
fig.savefig(FIG_DIR / "q3_release_decade.png")
plt.show()

In [ ]:
best = d.loc[d["avg_rating"].idxmax()]
worst = d.loc[d["avg_rating"].idxmin()]
old = con.sql('''
    SELECT CASE WHEN m.release_year < 1980 THEN 'Released before 1980'
                WHEN m.release_year >= 2000 THEN 'Released 2000 or later' END AS grp,
           count(*) AS n_ratings, round(avg(f.rating), 3) AS avg_rating
    FROM gold.fact_rating f JOIN gold.dim_movie m USING (movie_key)
    WHERE m.release_year < 1980 OR m.release_year >= 2000
    GROUP BY 1 ORDER BY 1 DESC
''').df()
display(old)
o = dict(zip(old["grp"], old["avg_rating"]))
display(Markdown(f'''
**Evidence:**
- Highest-rated decade: **{best.release_decade}** at **{best.avg_rating:.3f}** stars ({best.diff_vs_overall:+.3f} vs overall).
- Lowest-rated decade: **{worst.release_decade}** at **{worst.avg_rating:.3f}** ({worst.diff_vs_overall:+.3f} vs overall).
- Films released before 1980 average **{o.get("Released before 1980", float("nan")):.3f}**; films released 2000 or later
  average **{o.get("Released 2000 or later", float("nan")):.3f}**.
'''))

**Interpretation.** > ✏️ TODO after the full run: 2–4 sentences explaining what the evidence above means for the question.

---
## Q4 · How do casual, regular, heavy and power users differ?

For each activity segment: how many users, what share of all ratings they supply, and how generously they rate.

In [ ]:
Q4 = '''
SELECT u.activity_segment,
       count(DISTINCT u.user_key)                                   AS n_users,
       round(100.0 * count(DISTINCT u.user_key)
             / (SELECT count(*) FROM gold.dim_user), 1)             AS pct_of_users,
       count(*)                                                     AS n_ratings,
       round(100.0 * count(*) / sum(count(*)) OVER (), 1)           AS pct_of_ratings,
       round(avg(f.rating), 3)                                      AS avg_rating,
       round(stddev_samp(f.rating), 3)                              AS sd_rating,
       round(100 * avg(f.is_positive), 1)                           AS pct_positive
FROM gold.fact_rating f
JOIN gold.dim_user u ON u.user_key = f.user_key
GROUP BY u.activity_segment, u.segment_order
ORDER BY u.segment_order
'''
q4 = con.sql(Q4).df()
display(q4)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [3, 2], "wspace": 0.45})
y = np.arange(len(q4))[::-1]
h = 0.36
ax1.barh(y + h / 2, q4["pct_of_users"], height=h, color=BLUE, label="% of users")
ax1.barh(y - h / 2, q4["pct_of_ratings"], height=h, color=ORANGE, label="% of ratings")
for yi, (pu, pr) in zip(y, zip(q4["pct_of_users"], q4["pct_of_ratings"])):
    ax1.text(pu + 0.8, yi + h / 2, f"{pu:.1f}%", va="center", fontsize=8, color=INK_SECONDARY)
    ax1.text(pr + 0.8, yi - h / 2, f"{pr:.1f}%", va="center", fontsize=8, color=INK_SECONDARY)
ax1.set_yticks(y, q4["activity_segment"])
ax1.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax1.set_title("Share of users vs share of ratings")
ax1.legend(loc="lower right")
ax1.grid(axis="y", visible=False)

ax2.plot(q4["avg_rating"], y, "o", color=BLUE, markersize=9)
ax2.axvline(overall.avg_rating, color=INK_SECONDARY, linestyle="--", linewidth=1)
for yi, v in zip(y, q4["avg_rating"]):
    ax2.text(v, yi + 0.2, f"{v:.2f}", ha="center", fontsize=8, color=INK_SECONDARY)
ax2.set_yticks(y, [""] * len(y))
ax2.tick_params(axis="y", length=0)
ax2.set_title("Average rating (stars)")
ax2.set_xlabel("dashed line = overall average; axis does not start at 0")
ax2.grid(axis="y", visible=False)
fig.savefig(FIG_DIR / "q4_user_segments.png")
plt.show()

In [ ]:
heavy_plus = q4[q4["activity_segment"].str.startswith(("Heavy", "Power"))]
gen = q4.loc[q4["avg_rating"].idxmax()]
harsh = q4.loc[q4["avg_rating"].idxmin()]
display(Markdown(f'''
**Evidence:**
- Heavy + Power users are **{heavy_plus.pct_of_users.sum():.1f}%** of users but supply
  **{heavy_plus.pct_of_ratings.sum():.1f}%** of all ratings.
- Most generous segment: **{gen.activity_segment}** ({gen.avg_rating:.3f} stars, {gen.pct_positive:.1f}% positive);
  harshest: **{harsh.activity_segment}** ({harsh.avg_rating:.3f} stars, {harsh.pct_positive:.1f}% positive).
- Spread of ratings (std. dev.) ranges from {q4.sd_rating.min():.3f} to {q4.sd_rating.max():.3f} stars across segments.
'''))

**Interpretation.** > ✏️ TODO after the full run: 2–4 sentences explaining what the evidence above means for the question.

In [ ]:
con.close()
print("Analysis complete. Figures saved to", FIG_DIR)